# Fine-tune ViT5-base một lần trên dữ liệu y khoa

Notebook này thực hiện đúng một quy trình: kiểm tra dữ liệu y khoa, nạp trực tiếp `VietAI/vit5-base`, fine-tune trên tập train, chọn checkpoint tốt nhất bằng ROUGE-L của tập validation và đánh giá một lần trên tập test. Không nạp checkpoint từ lần huấn luyện cũ. Chế độ smoke test được bật hoặc tắt thủ công ở ô đầu tiên và hoạt động giống nhau trên local, Kaggle và Colab.

## 0. Chọn chế độ chạy

Đặt `SMOKE_TEST = True` để kiểm tra toàn bộ luồng bằng 1 bước huấn luyện. Đặt `False` để huấn luyện đầy đủ. `ESTIMATE_RUNTIME = True` sẽ benchmark ngắn và cảnh báo nếu tổng thời gian dự kiến vượt 12 giờ.

In [ ]:
SMOKE_TEST = True
ESTIMATE_RUNTIME = True

# Để None, notebook sẽ tự tìm dữ liệu. Có thể điền đường dẫn nếu Kaggle có nhiều dataset.
MEDICAL_DATA_DIR = None

## 1. Chuẩn bị mã nguồn và môi trường

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
IS_LOCAL = not IS_KAGGLE and not IS_COLAB
REPO_URL = "https://github.com/dungcony/y_khoa_sumarization.git"

def is_project(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (path / "src").is_dir()

search_roots = [Path.cwd(), *Path.cwd().parents]
search_roots.extend(path.parent for path in Path.cwd().glob("*/pyproject.toml"))
if IS_KAGGLE:
    search_roots.extend(path.parent for path in Path("/kaggle/input").rglob("pyproject.toml"))

project_root = next((path.resolve() for path in search_roots if is_project(path)), None)
if project_root is None:
    work_root = Path("/kaggle/working" if IS_KAGGLE else "/content") / "y_khoa_sumarization"
    if not work_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(work_root)], check=True)
    project_root = work_root.resolve()
elif IS_KAGGLE and str(project_root).startswith("/kaggle/input/"):
    work_root = Path("/kaggle/working/y_khoa_sumarization")
    work_root.mkdir(parents=True, exist_ok=True)
    for name in ("src", "scripts", "configs", "pyproject.toml"):
        source = project_root / name
        target = work_root / name
        if source.is_dir():
            shutil.copytree(source, target, dirs_exist_ok=True)
        elif source.is_file():
            shutil.copy2(source, target)
    project_root = work_root.resolve()

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Thư mục dự án: {project_root}")
print(f"Chế độ: {'SMOKE TEST' if SMOKE_TEST else 'HUẤN LUYỆN ĐẦY ĐỦ'}")

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print("Đã cài đặt thư viện của dự án.")

## 2. Xác định và kiểm tra dữ liệu y khoa

Mỗi thư mục dữ liệu phải có `train.csv`, `validation.csv`, `test.csv`; mỗi tệp phải có hai cột `article` và `summary`. Notebook tự nhận `/kaggle/input/medical_summarization`; nếu Kaggle chứa nhiều bộ dữ liệu, hãy điền `MEDICAL_DATA_DIR` ở ô cấu hình đầu tiên.

In [ ]:
import hashlib
import json
import platform
import time
import zipfile

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display

REQUIRED_FILES = ("train.csv", "validation.csv", "test.csv")

def valid_data_dir(path: Path) -> bool:
    if not all((path / name).is_file() for name in REQUIRED_FILES):
        return False
    try:
        return all(
            {"article", "summary"}.issubset(pd.read_csv(path / name, nrows=1).columns)
            for name in REQUIRED_FILES
        )
    except Exception:
        return False

configured_dir = MEDICAL_DATA_DIR or os.environ.get("MEDICAL_DATA_DIR")
candidates = []
if configured_dir:
    candidates.append(Path(configured_dir))
candidates.extend([project_root / "data", project_root / "tuan 5-6" / "data"])
if IS_KAGGLE:
    candidates.extend(path.parent for path in Path("/kaggle/input").rglob("train.csv"))
if IS_COLAB:
    candidates.append(Path("/content/data"))

valid_candidates = []
for path in candidates:
    resolved = path.expanduser().resolve()
    if resolved not in valid_candidates and valid_data_dir(resolved):
        valid_candidates.append(resolved)

if not valid_candidates:
    raise FileNotFoundError(
        "Không tìm thấy bộ ba train.csv, validation.csv và test.csv có cột article/summary. "
        "Hãy gắn bộ dữ liệu hoặc đặt MEDICAL_DATA_DIR."
    )
if len(valid_candidates) > 1 and not configured_dir:
    choices = "\n- ".join(str(path) for path in valid_candidates)
    raise RuntimeError(
        "Tìm thấy nhiều thư mục dữ liệu. Hãy đặt MEDICAL_DATA_DIR thành một trong các đường dẫn sau:\n- "
        + choices
    )

data_dir = valid_candidates[0]
print(f"Dữ liệu y khoa: {data_dir}")

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def normalize_article(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.lower()
    )

frames = {}
article_sets = {}
manifest_splits = {}
for split in ("train", "validation", "test"):
    path = data_dir / f"{split}.csv"
    frame = pd.read_csv(path, keep_default_na=False)
    frame = frame[["article", "summary"]].astype(str)
    empty_rows = int(
        ((frame["article"].str.strip() == "") | (frame["summary"].str.strip() == "")).sum()
    )
    normalized_articles = normalize_article(frame["article"])
    duplicate_articles = int(normalized_articles.duplicated().sum())
    if empty_rows or duplicate_articles:
        raise ValueError(
            f"{split}: {empty_rows} dòng rỗng, {duplicate_articles} bài viết trùng. "
            "Cần làm sạch dữ liệu trước khi huấn luyện."
        )
    frames[split] = frame
    article_sets[split] = set(normalized_articles)
    manifest_splits[split] = {
        "file": str(path),
        "sha256": sha256(path),
        "rows": len(frame),
        "article_words_mean": round(frame["article"].str.split().str.len().mean(), 2),
        "article_words_median": round(frame["article"].str.split().str.len().median(), 2),
        "summary_words_mean": round(frame["summary"].str.split().str.len().mean(), 2),
        "summary_words_median": round(frame["summary"].str.split().str.len().median(), 2),
    }

overlap = {
    "train_validation": len(article_sets["train"] & article_sets["validation"]),
    "train_test": len(article_sets["train"] & article_sets["test"]),
    "validation_test": len(article_sets["validation"] & article_sets["test"]),
}
if any(overlap.values()):
    raise ValueError(f"Phát hiện bài viết xuất hiện ở nhiều tập: {overlap}")

dataset_manifest = {
    "domain": "Vietnamese medical news",
    "columns": ["article", "summary"],
    "splits": manifest_splits,
    "cross_split_article_overlap": overlap,
}
display(pd.DataFrame(manifest_splits).T)
display(frames["train"].head(3))

## 3. Cấu hình một lần huấn luyện

Huấn luyện đầy đủ dùng 3 epoch, đánh giá và lưu checkpoint sau mỗi epoch. Smoke test dùng 4 mẫu train, 2 mẫu validation, 2 mẫu test và đúng 1 bước huấn luyện; chế độ này chỉ lưu metrics/dự đoán, không lưu trọng số hoặc checkpoint lớn. Tập test chưa được dùng ở bước chọn mô hình. Đổi `SMOKE_TEST` tại ô đầu tiên rồi chọn **Run All** để chuyển chế độ trên bất kỳ môi trường nào.

In [ ]:
from src.config import apply_overrides, config_to_dict, load_config
from src.utils import detect_precision, get_device_info, save_json, set_seed

if not SMOKE_TEST and not torch.cuda.is_available():
    raise RuntimeError("Cần bật GPU trước khi fine-tune ViT5-base.")

config_path = project_root / "configs" / "vit5_base_medical.yaml"
output_base = Path("/kaggle/working" if IS_KAGGLE else "/content" if IS_COLAB else project_root)
if SMOKE_TEST:
    smoke_run_id = time.strftime("smoke_%Y%m%d_%H%M%S")
    output_dir = (output_base / "outputs_smoke" / "vit5_base" / smoke_run_id).resolve()
else:
    output_dir = (output_base / "outputs_medical" / "vit5_base").resolve()
previous_run_markers = [
    output_dir / "trainer_state.json",
    output_dir / "best",
    *output_dir.glob("checkpoint-*"),
]
if any(path.exists() for path in previous_run_markers):
    raise FileExistsError(
        f"{output_dir} đã chứa kết quả huấn luyện. Đổi tên hoặc di chuyển thư mục này "
        "nếu thực sự cần chạy một thí nghiệm mới."
    )
output_dir.mkdir(parents=True, exist_ok=True)

config = load_config(config_path)
selected_precision = (
    detect_precision()
    if SMOKE_TEST or config.training.precision == "auto"
    else config.training.precision
)
overrides = {
    "data.train_file": str(data_dir / "train.csv"),
    "data.valid_file": str(data_dir / "validation.csv"),
    "data.test_file": str(data_dir / "test.csv"),
    "training.output_dir": str(output_dir),
    "training.precision": selected_precision,
}
if SMOKE_TEST:
    overrides.update({
        "data.max_train_samples": 4,
        "data.max_eval_samples": 2,
        "data.max_source_length": 128,
        "data.max_target_length": 64,
        "training.num_train_epochs": 1,
        "training.max_steps": 1,
        "training.per_device_train_batch_size": 1,
        "training.per_device_eval_batch_size": 1,
        "training.gradient_accumulation_steps": 1,
        "training.optim": "adamw_torch",
        "training.label_smoothing_factor": 0.0,
        "training.gradient_checkpointing": False,
        "training.eval_strategy": "steps",
        "training.eval_steps": 1,
        "training.save_strategy": "no",
        "training.save_total_limit": 1,
        "training.load_best_model_at_end": False,
        "training.early_stopping_patience": 0,
        "training.logging_steps": 1,
        "generation.max_length": 64,
        "generation.min_length": 5,
    })
else:
    # Cấu hình đã kiểm chứng phù hợp với Kaggle T4 x2. Ghi đè trực tiếp để
    # notebook không vô tình dùng YAML cũ còn nằm trong bản clone.
    overrides.update({
        "data.max_source_length": 768,
        "data.max_target_length": 160,
        "training.num_train_epochs": 3,
        "training.max_steps": -1,
        "training.per_device_train_batch_size": 4,
        "training.per_device_eval_batch_size": 4,
        "training.gradient_accumulation_steps": 4,
        "training.precision": "fp16",
        "training.optim": "adafactor",
        "training.gradient_checkpointing": False,
        "training.eval_strategy": "epoch",
        "training.save_strategy": "epoch",
        "training.load_best_model_at_end": True,
        "training.early_stopping_patience": 2,
        "generation.max_length": 160,
        "generation.min_length": 10,
        "generation.num_beams": 2,
    })
config = apply_overrides(config, overrides)
set_seed(config.training.seed)

device_info = get_device_info()
environment = {
    **device_info,
    "smoke_test": SMOKE_TEST,
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "transformers": str(transformers.__version__),
    "gpu_memory_gb": [
        round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2)
        for i in range(torch.cuda.device_count())
    ],
}
save_json(dataset_manifest, output_dir / "dataset_manifest.json")
save_json(environment, output_dir / "environment.json")
save_json(config_to_dict(config), output_dir / "resolved_config.json")
print(json.dumps(config_to_dict(config), ensure_ascii=False, indent=2))
print(json.dumps(environment, ensure_ascii=False, indent=2))

## 4. Nạp ViT5-base và mã hóa dữ liệu

In [ ]:
from src.data import load_dataset_from_files, preprocess_for_seq2seq
from src.model import enable_gradient_checkpointing, load_model, load_tokenizer
from src.utils import count_parameters

tokenizer = load_tokenizer(config.model)
model = load_model(config.model, tokenizer, config.generation)
# Không trả past_key_values khi train; đặc biệt quan trọng với DataParallel T4 x2.
model.config.use_cache = False
if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)

raw_dataset = load_dataset_from_files(
    train_file=config.data.train_file,
    valid_file=config.data.valid_file,
    test_file=config.data.test_file,
)
tokenized_dataset = preprocess_for_seq2seq(raw_dataset, tokenizer, config.data)
parameter_counts = count_parameters(model)
save_json(parameter_counts, output_dir / "model_parameters.json")
print(parameter_counts)

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
)

from src.callbacks import TrainingProgressCallback
from src.metrics import build_compute_metrics
from src.training_args import build_training_args

training_args = build_training_args(config)
if SMOKE_TEST:
    training_args.report_to = []
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)
callbacks = [
    TrainingProgressCallback(
        label="MEDICAL SMOKE TEST" if SMOKE_TEST else "MEDICAL FULL RUN",
        log_every_steps=config.training.logging_steps,
        heartbeat_seconds=60,
        log_file=output_dir / "training_progress.log",
    )
]
if not SMOKE_TEST:
    callbacks.append(EarlyStoppingCallback(
        early_stopping_patience=config.training.early_stopping_patience
    ))

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(tokenizer),
    callbacks=callbacks,
)
effective_batch_size = (
    config.training.per_device_train_batch_size
    * config.training.gradient_accumulation_steps
    * max(torch.cuda.device_count(), 1)
)
print(f"Kích thước lô hiệu dụng: {effective_batch_size}")
used_rows = {split: len(tokenized_dataset[split]) for split in ("train", "validation", "test")}
print(f"Số mẫu toàn bộ: {len(raw_dataset['train'])}/"
      f"{len(raw_dataset['validation'])}/{len(raw_dataset['test'])}")
print(f"Số mẫu dùng trong lần chạy này: {used_rows['train']}/"
      f"{used_rows['validation']}/{used_rows['test']}")

## 5. Ước lượng thời gian chạy

Ô dưới đây benchmark vài batch train mà không cập nhật trọng số và sinh thử trên một phần nhỏ validation. Ước lượng bao gồm train, các lượt validation, test, lưu checkpoint và 15% dự phòng. Có thể đặt `ESTIMATE_RUNTIME = False` ở ô đầu tiên để bỏ qua.

In [ ]:
import contextlib
import gc
import math

def estimate_full_runtime():
    train_loader = trainer.get_train_dataloader()
    benchmark_batches = min(4, len(train_loader))
    benchmark_model = model
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        benchmark_model = torch.nn.DataParallel(model)
    benchmark_model.train()

    amp_enabled = torch.cuda.is_available() and config.training.precision in {"fp16", "bf16"}
    amp_dtype = torch.float16 if config.training.precision == "fp16" else torch.bfloat16
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled and amp_dtype == torch.float16)
    batch_times = []

    for batch_index, batch in enumerate(train_loader):
        if batch_index >= benchmark_batches:
            break
        batch = {
            key: value.to(training_args.device) if isinstance(value, torch.Tensor) else value
            for key, value in batch.items()
        }
        model.zero_grad(set_to_none=True)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        batch_started_at = time.perf_counter()
        amp_context = (
            torch.autocast(device_type="cuda", dtype=amp_dtype)
            if amp_enabled
            else contextlib.nullcontext()
        )
        with amp_context:
            loss = benchmark_model(**batch).loss.mean()
        scaler.scale(loss).backward()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - batch_started_at
        if batch_index > 0 or benchmark_batches == 1:
            batch_times.append(elapsed)

    model.zero_grad(set_to_none=True)
    del batch, loss, scaler, benchmark_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    set_seed(config.training.seed)

    seconds_per_train_batch = float(np.mean(batch_times))
    if config.training.max_steps > 0:
        optimizer_steps = config.training.max_steps
        total_train_batches = optimizer_steps * config.training.gradient_accumulation_steps
    else:
        updates_per_epoch = max(
            len(train_loader) // config.training.gradient_accumulation_steps, 1
        )
        optimizer_steps = math.ceil(updates_per_epoch * config.training.num_train_epochs)
        total_train_batches = math.ceil(len(train_loader) * config.training.num_train_epochs)
    estimated_training_seconds = seconds_per_train_batch * total_train_batches * 1.10

    eval_samples = min(8, len(tokenized_dataset["validation"]))
    eval_subset = tokenized_dataset["validation"].select(range(eval_samples))
    model.config.use_cache = True
    eval_benchmark = trainer.predict(eval_subset, metric_key_prefix="estimate")
    eval_runtime = float(eval_benchmark.metrics["estimate_runtime"])
    seconds_per_eval_sample = eval_runtime / eval_samples
    model.config.use_cache = False
    del eval_benchmark, eval_subset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    strategy = str(training_args.eval_strategy).split(".")[-1].lower()
    if strategy == "epoch":
        scheduled_validations = math.ceil(config.training.num_train_epochs)
    elif strategy == "steps":
        scheduled_validations = optimizer_steps // config.training.eval_steps
    else:
        scheduled_validations = 0
    total_eval_samples = (
        used_rows["validation"] * (scheduled_validations + 1)
        + used_rows["test"]
    )
    estimated_evaluation_seconds = seconds_per_eval_sample * total_eval_samples

    save_strategy = str(training_args.save_strategy).split(".")[-1].lower()
    if save_strategy == "epoch":
        save_events = math.ceil(config.training.num_train_epochs)
    elif save_strategy == "steps":
        save_events = optimizer_steps // config.training.save_steps
    else:
        save_events = 0
    estimated_save_seconds = save_events * 180 + (0 if SMOKE_TEST else 180)
    estimated_total_seconds = (
        estimated_training_seconds
        + estimated_evaluation_seconds
        + estimated_save_seconds
    ) * 1.15

    runtime_estimate = {
        "smoke_test": SMOKE_TEST,
        "benchmark_train_batches": benchmark_batches,
        "benchmark_eval_samples": eval_samples,
        "seconds_per_train_batch": round(seconds_per_train_batch, 3),
        "seconds_per_eval_sample": round(seconds_per_eval_sample, 3),
        "optimizer_steps": optimizer_steps,
        "estimated_training_hours": round(estimated_training_seconds / 3600, 2),
        "estimated_evaluation_hours": round(estimated_evaluation_seconds / 3600, 2),
        "estimated_saving_hours": round(estimated_save_seconds / 3600, 2),
        "estimated_total_hours_with_15_percent_buffer": round(estimated_total_seconds / 3600, 2),
        "fits_kaggle_12h": estimated_total_seconds <= 12 * 3600,
    }
    save_json(runtime_estimate, output_dir / "runtime_estimate.json")
    display(pd.DataFrame([runtime_estimate]).T.rename(columns={0: "Giá trị"}))
    if not runtime_estimate["fits_kaggle_12h"]:
        print("CẢNH BÁO: cấu hình được ước lượng vượt giới hạn 12 giờ của Kaggle.")
    else:
        print("Ước lượng nằm trong giới hạn 12 giờ của Kaggle.")
    return runtime_estimate

runtime_estimate = estimate_full_runtime() if ESTIMATE_RUNTIME else None

## 6. Fine-tune đúng một lần

Chỉ chạy ô dưới đây khi dữ liệu, thiết bị và cấu hình đã đúng. Huấn luyện đầy đủ cần GPU; smoke test có thể chạy trên CPU. Mỗi smoke test được lưu trong một thư mục có dấu thời gian riêng, còn lần huấn luyện đầy đủ vẫn được bảo vệ khỏi ghi đè.

In [ ]:
import gc

model.config.use_cache = False
model.zero_grad(set_to_none=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

started_at = time.time()
train_result = trainer.train()
training_seconds = time.time() - started_at

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

model.config.use_cache = True
best_model_dir = None
if not SMOKE_TEST:
    best_model_dir = output_dir / "best"
    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(best_model_dir)

validation_metrics = trainer.evaluate(
    eval_dataset=tokenized_dataset["validation"],
    metric_key_prefix="validation",
)
trainer.log_metrics("validation", validation_metrics)
save_json(validation_metrics, output_dir / "validation_metrics.json")
print(f"Thời gian fine-tune: {training_seconds / 3600:.2f} giờ")

## 7. Đánh giá tập test và xuất kết quả

Tập test chỉ được đánh giá sau khi mô hình tốt nhất theo validation đã được chọn. Tệp JSONL và CSV đều chứa văn bản nguồn, tóm tắt tham chiếu và tóm tắt do mô hình sinh ra để phân tích định tính trong báo cáo.

In [ ]:
from src.evaluation_io import export_predictions

test_started_at = time.time()
test_output = trainer.predict(
    tokenized_dataset["test"],
    metric_key_prefix="test",
)
test_seconds = time.time() - test_started_at
test_metrics = dict(test_output.metrics)
trainer.log_metrics("test", test_metrics)
save_json(test_metrics, output_dir / "test_metrics.json")

jsonl_path = output_dir / "test_predictions.jsonl"
export_predictions(
    predictions=test_output.predictions,
    labels=test_output.label_ids,
    tokenizer=tokenizer,
    dataset=raw_dataset["test"],
    output_path=jsonl_path,
)
prediction_rows = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines()]
predictions_csv = output_dir / "test_predictions.csv"
pd.DataFrame(prediction_rows).to_csv(predictions_csv, index=False)

run_summary = {
    "smoke_test": SMOKE_TEST,
    "model": config.model.name_or_path,
    "dataset_domain": dataset_manifest["domain"],
    "dataset_rows": {split: len(raw_dataset[split]) for split in raw_dataset},
    "used_rows": used_rows,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_metric": trainer.state.best_metric,
    "training_seconds": round(training_seconds, 2),
    "test_seconds": round(test_seconds, 2),
    "test_samples_per_second": round(used_rows["test"] / test_seconds, 4),
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
}
save_json(run_summary, output_dir / "run_summary.json")
display(pd.DataFrame(prediction_rows).head(10))
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

## 8. Đóng gói tệp dùng cho báo cáo

Tệp ZIP chỉ chứa cấu hình, dấu vết dữ liệu, chỉ số và dự đoán. Huấn luyện đầy đủ lưu trọng số trong `best` và các `checkpoint-*`; smoke test không tạo các thư mục lớn này.

In [ ]:
artifact_names = [
    "dataset_manifest.json",
    "environment.json",
    "model_parameters.json",
    "resolved_config.json",
    "train_results.json",
    "validation_metrics.json",
    "test_metrics.json",
    "trainer_state.json",
    "training_progress.log",
    "test_predictions.jsonl",
    "test_predictions.csv",
    "run_summary.json",
    "runtime_estimate.json",
]
archive_path = output_dir / "report_artifacts.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in artifact_names:
        path = output_dir / name
        if path.is_file():
            archive.write(path, arcname=name)

if SMOKE_TEST:
    print("Smoke test hoàn tất; không lưu trọng số hoặc checkpoint.")
else:
    print(f"Mô hình tốt nhất: {best_model_dir}")
print(f"Tệp dùng cho báo cáo: {archive_path}")